In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyID string"
        #     ",SourceSystem string"
        #     ",PortfolioKey string"
        #     ",PortfolioName string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",CDI string"
        #     ",RepresentedByPartyKey string"
        #     ",RepresentedByPartyName string"
        #     ",RepresentedByPartyType string"
        #     ",PartyKey string"
        #     ",PartyRepresentKey string"
        #     ",PartyTypeCode string"
        #     ",PartyType string"
        #     ",PartyStatusCode string"
        #     ",PartyStatus string"
        #     ",PartyStatusDate date"
        #     ",L3869StatusCode string"
        #     ",L3869Status string"
        #     ",L3869CourtDate timestamp"
        #     ",OCWStatus string"
        #     ",DeceasedFlag boolean"
        #     ",L3869InterbankingStatusCode int"
        #     ",L3869InterBankingStatus string"
        #     ",Watchlist string"
        #     ",FirstNameGR string"
        #     ",FirstNameENG string"
        #     ",LastNameGR string"
        #     ",LastNameENG string"
        #     ",FirstFatherNameGR string"
        #     ",FirstFatherNameENG string"
        #     ",LastFatherNameGR string"
        #     ",FirstMotherNameGR string"
        #     ",LastFatherNameENG string"
        #     ",LastMotherNameGR string"
        #     ",LastMotherNameENG string"
        #     ",LegalNameGR string"
        #     ",LegalNameENG string"
        #     ",PrimaryCommercialGR string"
        #     ",PrimaryCommercialENG string"
        #     ",SalutationTypeCode string"
        #     ",SalutationType string"
        #     ",GenderCode string"
        #     ",Gender string"
        #     ",MaritalStatusCode string"
        #     ",MaritalStatus string"
        #     ",CitizenshipCode string"
        #     ",Citizenship string"
        #     ",NationalityCode string"
        #     ",Nationality string"
        #     ",OccupationCode string"
        #     ",Occupation string"
        #     ",BirthDate string"
        #     ",BirthPlace string"
        #     ",DeathDate date"
        #     ",CommunicationLanguageCode int"
        #     ",CommunicationLanguage string"
        #     ",TaxRegistrationNumber string"
        #     ",TaxResidenceCountry string"
        #     ",OrganizationOriginCountry string"
        #     ",PassportNumber string"
        #     ",PassportIssuingAuthority string"
        #     ",PassportIssueDate date"
        #     ",PassportExpiredDate date"
        #     ",NationalNumber string"
        #     ",NationalNumberIssuingAuthority string"
        #     ",NationalNumberIssueDate date"
        #     ",NationalNumberExpiredDate date"
        #     ",MilitaryNumber string"
        #     ",MilitaryAuthority string"
        #     ",MilitaryIssueDate date"
        #     ",MilitaryExpiredDate date"
        #     ",SocialSecurityNumber string"
        #     ",ChildrenCertification string"
        #     ",ChildrenCertificationIssuingAuthority string"
        #     ",EnterpriseStartDate string"
        #     ",EnterpriseEndDate date"
        #     ",EconomicActivityCode int"
        #     ",EconomicActivity string"
        #     ",BranchActivityCode int"
        #     ",BranchActivity string"
        #     ",NumberOfChildren string"
        #     ",GeneralCommercialRegister string"
        #     ",NumberofEmployees string"
        #     ",TotalIncome string"
        #     ",BalanceSheetTotalAmount string"
        #     ",AnnualTurnoverAmount string"
        #     ",GroupCode string"
        #     ",GroupName string"
        #     ",GroupCodePortfolio int"
        #     ",PartyBankruptcyStage int"
        #     ",CovidFlag string"
        #     ",CovidCategoryCode int"
        #     ",CovidCategory string"
        #     ",AffectedCategoryCode int"
        #     ",AffectedCategory string"
        #     ",MonitoringUnitCode string"
        #     ",MonitoringUnit string"
        #     ",CreditRatingTypeCode int"
        #     ",CreditRatingType string"
        #     ",EducationLevelCode int"
        #     ",EducationLevel string"
        #     ",BalanceCategoryCode string"
        #     ",BalanceCategory string"
        #     ",ExtraSegmentCode string"
        #     ",ExtraSegment string"
        #     ",SegmentCode string"
        #     ",Segment string"
        #     ",SubSegmentCode string"
        #     ",SubSegment string"
        #     ",LegalFormCode int"
        #     ",LegalForm string"
        #     ",IsWholeshaleFlag boolean"
        #     ",EmploymentStatusCode int"
        #     ",EmploymentStatus string"
        #     ",EmploymentStatusDate date"
        #     ",CreatedBy string"
        #     ",CreatedByDate timestamp"
        #     ",LastUpdatedBy string"
        #     ",LastUpdatedByDate timestamp"
        #     ",ResubmissionStatus string"
        #     ",N4605Flag int"
        #     ",ConnectedBorrowerFlag boolean"
        #     ",StaffFlag string"
        #     ",L3869BorrowerHasOver50KBalance boolean"
        #     ",TargetLegalActionCode int"
        #     ",TargetLegalAction string"
        #     ",SubServicer string"
        #     ",RiskDescription string"
        #     ",MobileTel string"
        #     ",MobileTel2 string"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Party"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "Prefix",
            "PortfolioName",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================================ party_arotron ==============

        party_arotron_extra_filters = None

        if party_arotron_extra_filters is not None:
            party_arotron_extra_filters.append(date_filter)
            party_arotron_final_filters = " AND ".join(party_arotron_extra_filters)
        else:
            party_arotron_final_filters = date_filter

        party_arotron_selected_fields = [
            "eaisig_bp_cust_s_unid",
            "eaisig_bp_cust_whl_ind",
            "eaisig_bp_cust_fname",
            "eaisig_bp_cust_latin",
            "eaisig_bp_cust_lname",
            "eaisig_bp_cust_fathername",
            "eaisig_bp_cust_mothname",
            "eaisig_bp_cust_title",
            "eaisig_bp_cust_sex",
            "eaisig_bp_cust_famstatus",
            "eaisig_bp_cust_citizenship",
            "eaisig_bp_cust_job",
            "eaisig_bp_cust_dob",
            "eaisig_bp_reg_date",
            "eaisig_bp_key",
            "eaisig_bp_cust_vat",
            "eaisig_bp_cust_passportnum",
            "eaisig_bp_cust_idnum",
            "eaisig_bp_cust_childnum",
            "eaisig_bp_numbers_of_employees",
            "eaisig_bp_cust_annual_income_amt",
            "eaiSIG_BP_CUST_GROUP",
            "eaisig_bp_cust_group_name",
            "businessdate",
        ]

        party_arotron = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_BP",
            party_arotron_final_filters,
            party_arotron_selected_fields,
        )

        # =============== Join with portfolio ===============
        party_arotron = party_arotron.join(
            portfolio,
            on=substring(party_arotron["eaisig_bp_cust_s_unid"], 1, 3) == portfolio["Prefix"],
            how="left"
        )

        party_arotron_primary_key_fields = [
            party_arotron.eaisig_bp_cust_s_unid,
            lit("arotron"),
        ]

        party_arotron = (
            party_arotron.withColumn(
                "PartyID", sha2(concat_ws(";", *party_arotron_primary_key_fields), 256)
            )
            .withColumn(
                "PartyType",
                when(party_arotron.eaisig_bp_cust_whl_ind == "N", "Individual").when(
                    party_arotron.eaisig_bp_cust_whl_ind == "Y", "Organization"
                ),
            )
            .withColumn(
                "Gender",
                when(party_arotron.eaisig_bp_cust_sex == 0, "Female").otherwise("Male"),
            )
            .withColumn(
                "BirthDate",
                when(
                    party_arotron.eaisig_bp_cust_dob.isNotNull(),
                    party_arotron.eaisig_bp_cust_dob,
                )
                .otherwise("")
                .cast("timestamp"),
            )
            .withColumn(
                "EnterpriseStartDate",
                when(
                    party_arotron.eaisig_bp_reg_date.isNotNull(),
                    party_arotron.eaisig_bp_reg_date,
                )
                .otherwise("")
                .cast("date"),
            )
            .withColumn("RepresentedByPartyType", lit(None).cast("string"))
            .withColumn("PartyStatusCode", lit(None).cast("string"))
            .withColumn("PartyStatusDate", lit(None).cast("date"))
            .withColumn("LastNameENG", lit(None).cast("string"))
            .withColumn("LegalNameGR", lit(None).cast("string"))
            .withColumn("PrimaryCommercialGR", lit(None).cast("string"))
            .withColumn("SalutationType", lit(None).cast("string"))
            .withColumn("EnterpriseEndDate", lit(None).cast("date"))
            .withColumn("MaritalStatus", lit(None).cast("string"))
            .withColumn("NationalityCode", lit(None).cast("string"))
            .withColumn("Nationality", lit(None).cast("string"))
            .withColumn("Occupation", lit(None).cast("string"))
            .withColumn("CommunicationLanguage", lit(None).cast("string"))
            .withColumn("EconomicActivityCode", lit(None).cast("int"))
            .withColumn("EconomicActivity", lit(None).cast("string"))
            .withColumn("BranchActivityCode", lit(None).cast("int"))
            .withColumn("BranchActivity", lit(None).cast("string"))
            .withColumn("EducationLevel", lit(None).cast("string"))
            .withColumn("MonitoringUnitCode", lit(None).cast("int"))
            .withColumn("EducationLevelCode", lit(None).cast("int"))
            .withColumn("LegalFormCode", lit(None).cast("int"))
            .withColumn("LegalForm", lit(None).cast("string"))
            .withColumn("EmploymentStatus", lit(None).cast("string"))
            .withColumn("PassportIssuingAuthority", lit(None).cast("string"))
            .withColumn("MilitaryAuthority", lit(None).cast("string"))
            .withColumn("ChildrenCertification", lit(None).cast("string"))
            .withColumn("SocialSecurityNumber", lit(None).cast("string"))
            .withColumn(
                "ChildrenCertificationIssuingAuthority", lit(None).cast("string")
            )
            .withColumn("PartyBankruptcyStage", lit(None).cast("int"))
            .withColumn("CovidFlag", lit(None).cast("int"))
            .withColumn("CovidCategoryCode", lit(None).cast("int"))
            .withColumn("BalanceCategoryCode", lit(None).cast("int"))
            .withColumn("ExtraSegmentCode", lit(None).cast("string"))
            .withColumn("CreditRatingTypeCode", lit(None).cast("int"))
            .withColumn("SubSegmentCode", lit(None).cast("string"))
            .withColumn("SegmentCode", lit(None).cast("string"))
            .withColumn("PartyTypeCode", lit(None).cast("string"))
            .withColumn("FirstNameENG", lit(None).cast("string"))
        )

        # ================== Selection of Final Columns ==================

        party_arotron = party_arotron.select(
            party_arotron.PartyID.alias("PartyID"),
            lit("arotron").cast("string").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            party_arotron.eaisig_bp_cust_s_unid.cast("string").alias("CDI"),
            lit(None).cast("string").alias("RepresentedByPartyKey"),
            lit(None).cast("string").alias("RepresentedByPartyName"),
            lit(None).cast("string").alias("RepresentedByPartyType"),
            party_arotron.eaisig_bp_cust_s_unid.cast("string").alias("PartyKey"),
            party_arotron.eaisig_bp_cust_s_unid.cast("string").alias(
                "PartyRepresentKey"
            ),
            party_arotron.eaisig_bp_cust_whl_ind.cast("string").alias("PartyTypeCode"),
            party_arotron.PartyType.cast("string").alias("PartyType"),
            lit(None).cast("string").alias("PartyStatusCode"),
            lit(None).cast("string").alias("PartyStatus"),
            lit(None).cast("date").alias("PartyStatusDate"),
            lit(None).alias("L3869StatusCode"),
            lit(None).alias("L3869Status"),
            lit(None).alias("L3869CourtDate"),
            lit(None).alias("OCWStatus"),
            lit(None).alias("DeceasedFlag"),
            lit(None).alias("L3869InterbankingStatusCode"),
            lit(None).alias("L3869InterBankingStatus"),
            lit(None).alias("Watchlist"),
            party_arotron.eaisig_bp_cust_fname.cast("string").alias("FirstNameGR"),
            party_arotron.eaisig_bp_cust_latin.cast("string").alias("FirstNameENG"),
            party_arotron.eaisig_bp_cust_lname.cast("string").alias("LastNameGR"),
            lit(None).cast("string").alias("LastNameENG"),
            party_arotron.eaisig_bp_cust_fathername.cast("string").alias(
                "FirstFatherNameGR"
            ),
            lit(None).cast("string").alias("FirstFatherNameENG"),
            lit(None).cast("string").alias("LastFatherNameGR"),
            lit(None).cast("string").alias("LastFatherNameENG"),
            party_arotron.eaisig_bp_cust_mothname.cast("string").alias("FirstMotherNameGR"),
            party_arotron.eaisig_bp_cust_mothname.cast("string").alias(
                "LastMotherNameGR"
            ),
            lit(None).cast("string").alias("LastMotherNameENG"),
            lit(None).cast("string").alias("LegalNameGR"),
            lit(None).cast("string").alias("LegalNameENG"),
            lit(None).cast("string").alias("PrimaryCommercialGR"),
            lit(None).cast("string").alias("PrimaryCommercialENG"),
            party_arotron.eaisig_bp_cust_title.cast("string").alias(
                "SalutationTypeCode"
            ),
            lit(None).cast("string").alias("SalutationType"),
            party_arotron.eaisig_bp_cust_sex.cast("string").alias("GenderCode"),
            party_arotron.Gender.cast("string").alias("Gender"),
            party_arotron.eaisig_bp_cust_famstatus.cast("string").alias(
                "MaritalStatusCode"
            ),
            lit(None).cast("string").alias("MaritalStatus"),
            party_arotron.eaisig_bp_cust_citizenship.cast("string").alias(
                "Citizenship"
            ),
            party_arotron.eaisig_bp_cust_citizenship.cast("string").alias(
                "CitizenshipCode"
            ),
            lit(None).cast("string").alias("NationalityCode"),
            lit(None).cast("string").alias("Nationality"),
            party_arotron.eaisig_bp_cust_job.cast("string").alias("OccupationCode"),
            party_arotron.eaisig_bp_cust_job.cast("string").alias("Occupation"),
            party_arotron.BirthDate.cast("string").alias("BirthDate"),
            lit(None).cast("string").alias("BirthPlace"),
            lit(None).cast("date").alias("DeathDate"),
            lit(None).cast("string").alias("CommunicationLanguage"),
            lit(None).cast("int").alias("CommunicationLanguageCode"),
            party_arotron.eaisig_bp_cust_vat.cast("string").alias(
                "TaxRegistrationNumber"
            ),
            lit(None).cast("string").alias("TaxResidenceCountry"),
            lit(None).cast("string").alias("OrganizationOriginCountry"),
            party_arotron.eaisig_bp_cust_passportnum.cast("string").alias(
                "PassportNumber"
            ),
            lit(None).cast("string").alias("PassportIssuingAuthority"),
            lit(None).cast("date").alias("PassportIssueDate"),
            lit(None).cast("date").alias("PassportExpiredDate"),
            party_arotron.eaisig_bp_cust_idnum.cast("string").alias("NationalNumber"),
            lit(None).cast("string").alias("NationalNumberIssuingAuthority"),
            lit(None).cast("date").alias("NationalNumberIssueDate"),
            lit(None).cast("date").alias("NationalNumberExpiredDate"),
            lit(None).cast("string").alias("MilitaryNumber"),
            lit(None).cast("string").alias("MilitaryAuthority"),
            lit(None).cast("date").alias("MilitaryIssueDate"),
            lit(None).cast("date").alias("MilitaryExpiredDate"),
            lit(None).cast("string").alias("SocialSecurityNumber"),
            lit(None).cast("string").alias("ChildrenCertification"),
            lit(None).cast("string").alias("ChildrenCertificationIssuingAuthority"),
            party_arotron.EnterpriseStartDate.cast("string").alias(
                "EnterpriseStartDate"
            ),
            lit(None).cast("date").alias("EnterpriseEndDate"),
            lit(None).cast("int").alias("EconomicActivityCode"),
            lit(None).cast("string").alias("EconomicActivity"),
            lit(None).cast("int").alias("BranchActivityCode"),
            lit(None).cast("string").alias("BranchActivity"),
            party_arotron.eaisig_bp_cust_childnum.cast("string").alias(
                "NumberOfChildren"
            ),
            lit(None).cast("string").alias("GeneralCommercialRegister"),
            party_arotron.eaisig_bp_numbers_of_employees.cast("string").alias(
                "NumberofEmployees"
            ),
            party_arotron.eaisig_bp_cust_annual_income_amt.cast("string").alias(
                "TotalIncome"
            ),
            lit(None).cast("string").alias("BalanceSheetTotalAmount"),
            lit(None).cast("string").alias("AnnualTurnoverAmount"),
            party_arotron.eaiSIG_BP_CUST_GROUP.cast("string").alias("GroupCode"),
            party_arotron.eaisig_bp_cust_group_name.cast("string").alias("GroupName"),
            lit(None).alias("GroupCodePortfolio"),
            lit(None).cast("int").alias("PartyBankruptcyStage"),
            lit(None).cast("string").alias("CovidFlag"),
            lit(None).cast("int").alias("CovidCategoryCode"),
            lit(None).cast("string").alias("CovidCategory"),
            lit(None).alias("AffectedCategoryCode"),
            lit(None).alias("AffectedCategory"),
            lit(None).cast("string").alias("MonitoringUnitCode"),
            lit(None).cast("string").alias("MonitoringUnit"),
            lit(None).cast("int").alias("CreditRatingTypeCode"),
            lit(None).cast("string").alias("CreditRatingType"),
            lit(None).cast("int").alias("EducationLevelCode"),
            lit(None).cast("string").alias("EducationLevel"),
            lit(None).cast("string").alias("BalanceCategoryCode"),
            lit(None).cast("string").alias("BalanceCategory"),
            lit(None).cast("string").alias("ExtraSegmentCode"),
            lit(None).cast("string").alias("ExtraSegment"),
            lit(None).cast("string").alias("SegmentCode"),
            lit(None).cast("string").alias("Segment"),
            lit(None).cast("string").alias("SubSegmentCode"),
            lit(None).cast("string").alias("SubSegment"),
            lit(None).cast("int").alias("LegalFormCode"),
            lit(None).cast("string").alias("LegalForm"),
            lit(None).alias("IsWholeshaleFlag"),
            lit(None).cast("string").alias("EmploymentStatus"),
            lit(None).alias("EmploymentStatusCode"),
            lit(None).cast("date").alias("EmploymentStatusDate"),
            lit(None).cast("string").alias("CreatedBy"),
            lit(None).cast("timestamp").alias("CreatedByDate"),
            lit(None).cast("string").alias("LastUpdatedBy"),
            lit(None).cast("timestamp").alias("LastUpdatedByDate"),
            lit(None).alias("ResubmissionStatus"),
            lit(None).alias("N4605Flag"),
            lit(None).alias("ConnectedBorrowerFlag"),
            lit(None).alias("StaffFlag"),
            lit(None).alias("L3869BorrowerHasOver50KBalance"),
            lit(None).alias("TargetLegalActionCode"),
            lit(None).alias("TargetLegalAction"),
            lit(None).cast("string").alias("SubServicer"),
            lit(None).cast("string").alias("RiskDescription"),
            lit(None).cast("string").alias("MobileTel"),
            lit(None).cast("string").alias("MobileTel2"),
            party_arotron.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        party_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:

load_curated_party_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)